# Football Player Network Analysis using Apache Spark

Professional notebook structure.

In [1]:
from pyspark.sql import SparkSession

sparkSession = SparkSession.builder.appName("FootballPlaystyleAnalysis").master("local[*]").getOrCreate()
spark = sparkSession
sparkSession

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/16 15:45:33 WARN Utils: Your hostname, Onur-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.100 instead (on interface en0)
26/07/16 15:45:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/16 15:45:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Veri Setlerini Okuma

In [2]:
events = spark.read.option("multiline", "true").json("statsbomb-open-data/data/events/*.json")

26/07/16 15:45:36 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: statsbomb-open-data/data/events/*.json.
java.io.FileNotFoundException: File statsbomb-open-data/data/events/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apac

In [3]:
##Events kontrolleri
events.printSchema()

events.count()

len(events.columns)

events.show(5, truncate=False)

root
 |-- 50_50: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- bad_behaviour: struct (nullable = true)
 |    |-- card: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_receipt: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_recovery: struct (nullable = true)
 |    |-- offensive: boolean (nullable = true)
 |    |-- recovery_failure: boolean (nullable = true)
 |-- block: struct (nullable = true)
 |    |-- deflection: boolean (nullable = true)
 |    |-- offensive: boolean (nullable = true)
 |    |-- save_block: boolean (nullable = true)
 |-- carry: struct (nullable = true)
 |    |-- end_location: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- clearan

26/07/16 15:46:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----+-------------+------------+-------------+-----+-----+---------+------------+-------+----+--------+--------------+--------+----------+--------+----------+------------------------------------+-----+---------------+------------+------------+------+----------+----------+----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+------------------+-------------------------+----------+-------------------------------+----------+---------------+--------------------------------------+------+----+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
##event types
from pyspark.sql.functions import col

events.select(col("type.name").alias("event_type")) \
      .distinct() \
      .orderBy("event_type") \
      .show(100, truncate=False)

+-----------------+
|event_type       |
+-----------------+
|50/50            |
|Bad Behaviour    |
|Ball Receipt*    |
|Ball Recovery    |
|Block            |
|Camera On        |
|Camera off       |
|Carry            |
|Clearance        |
|Dispossessed     |
|Dribble          |
|Dribbled Past    |
|Duel             |
|Error            |
|Foul Committed   |
|Foul Won         |
|Goal Keeper      |
|Half End         |
|Half Start       |
|Injury Stoppage  |
|Interception     |
|Miscontrol       |
|Offside          |
|Own Goal Against |
|Own Goal For     |
|Pass             |
|Player Off       |
|Player On        |
|Pressure         |
|Referee Ball-Drop|
|Shield           |
|Shot             |
|Starting XI      |
|Substitution     |
|Tactical Shift   |
+-----------------+



In [5]:
events.groupBy(
    col("type.name").alias("event_type")
).count().orderBy(col("count").desc()).show(100, truncate=False)

+-----------------+-------+
|event_type       |count  |
+-----------------+-------+
|Pass             |4103347|
|Ball Receipt*    |3844869|
|Carry            |3204878|
|Pressure         |1394727|
|Ball Recovery    |461754 |
|Duel             |310469 |
|Clearance        |196297 |
|Block            |168752 |
|Dribble          |146663 |
|Goal Keeper      |132397 |
|Miscontrol       |126295 |
|Foul Committed   |117581 |
|Foul Won         |111772 |
|Dispossessed     |110366 |
|Shot             |108282 |
|Interception     |96130  |
|Dribbled Past    |90736  |
|Substitution     |28044  |
|Injury Stoppage  |18172  |
|Half End         |17216  |
|Half Start       |17216  |
|50/50            |15838  |
|Tactical Shift   |11739  |
|Starting XI      |8470   |
|Referee Ball-Drop|6264   |
|Shield           |6007   |
|Player Off       |4541   |
|Player On        |4500   |
|Bad Behaviour    |2987   |
|Camera On        |2595   |
|Error            |2256   |
|Offside          |1513   |
|Camera off       |6

In [6]:
passes = events.filter(col("type.name") == "Pass")
passes.count()

4103347

In [7]:
## Extract the necessary columns for analysis
passes = passes.select(
    col("player.id").alias("passer_id"),
    col("player.name").alias("passer"),
    col("pass.recipient.id").alias("receiver_id"),
    col("pass.recipient.name").alias("receiver"),
    col("team.id").alias("team_id"),
    col("team.name").alias("team"),
    "minute",
    "second",
    "location",
    col("pass.end_location").alias("end_location")
)

##Show
passes.show(10, truncate=False)

+---------+-------------------------+-----------+-------------------------+-------+------+------+------+------------+------------+
|passer_id|passer                   |receiver_id|receiver                 |team_id|team  |minute|second|location    |end_location|
+---------+-------------------------+-----------+-------------------------+-------+------+------+------+------------+------------+
|5487     |Antoine Griezmann        |10481      |Aurélien Djani Tchouaméni|771    |France|0     |0     |[60.0, 40.0]|[48.4, 38.1]|
|10481    |Aurélien Djani Tchouaméni|24778      |Eduardo Camavinga        |771    |France|0     |2     |[47.9, 37.4]|[49.3, 28.7]|
|24778    |Eduardo Camavinga        |8519       |Dayotchanculle Upamecano |771    |France|0     |4     |[49.0, 25.2]|[38.1, 46.8]|
|8519     |Dayotchanculle Upamecano |3961       |N'Golo Kanté             |771    |France|0     |7     |[41.6, 49.4]|[49.5, 52.3]|
|3961     |N'Golo Kanté             |17592      |William Saliba           |771    |

In [8]:
passes.filter(col("receiver").isNull()).count()

256163

In [9]:
##Filtreyi uygula - alıcısı olmayan pasları çıkar
passes = passes.filter(col("receiver").isNotNull())
## Sayı kontrol
passes.count()

3847184

In [10]:
## create an edge list for the passes
edges = (
    passes.groupBy(
        "passer_id",
        "passer",
        "receiver_id",
        "receiver",
        "team_id",
        "team"
    )
    .count()
    .withColumnRenamed("count", "pass_count")

    
    
)

##kontrol
edges.show(20, truncate=False)

edges.count()

+---------+-----------------------------+-----------+------------------------------+-------+-------------------+----------+
|passer_id|passer                       |receiver_id|receiver                      |team_id|team               |pass_count|
+---------+-----------------------------+-----------+------------------------------+-------+-------------------+----------+
|8519     |Dayotchanculle Upamecano     |4445       |Jules Koundé                  |771    |France             |77        |
|5204     |Bruno Miguel Borges Fernandes|41092      |Nuno Mendes                   |780    |Portugal           |39        |
|3961     |N'Golo Kanté                 |3009       |Kylian Mbappé Lottin          |771    |France             |21        |
|10595    |Raphael Dias Belloli         |3063       |Danilo Luiz da Silva          |781    |Brazil             |20        |
|30486    |Pedro González López         |5203       |Sergio Busquets i Burgos      |772    |Spain              |60        |
|16022  

208518

In [11]:
## creating verteices

vertices = (
    passes.select(
        col("passer_id").alias("id"),
        col("passer").alias("name"),
        "team_id",
        "team"
    )
    .distinct()
)

vertices.count()
vertices.show(20, truncate=False)

+-----+--------------------------------+-------+------------------------+
|id   |name                            |team_id|team                    |
+-----+--------------------------------+-------+------------------------+
|3009 |Kylian Mbappé Lottin            |131    |Paris Saint-Germain     |
|6840 |Marcos Llorente Moreno          |772    |Spain                   |
|23725|Roman Bezus                     |911    |Ukraine                 |
|5487 |Antoine Griezmann               |212    |Atlético Madrid         |
|34639|Vitor Machado Ferreira          |131    |Paris Saint-Germain     |
|48396|Rocco Reitz                     |185    |Borussia Mönchengladbach|
|13620|Éder Gabriel Militão            |781    |Brazil                  |
|25305|Pedro Guilherme Abreu dos Santos|781    |Brazil                  |
|18618|Serhiy Kryvtsov                 |911    |Ukraine                 |
|31900|Oleksandr Karavaev              |911    |Ukraine                 |
|11396|Florian Grillitsch             

Matches'i okuma işlemi

In [12]:
matches = spark.read.option(
    "multiline", "true"
).json("statsbomb-open-data/data/matches/*/*.json")

26/07/16 15:47:22 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: statsbomb-open-data/data/matches/*/*.json.
java.io.FileNotFoundException: File statsbomb-open-data/data/matches/*/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at or

In [13]:
## Match kontrolleri

matches.select(
    "match_id",
    "competition.competition_name",
    "season.season_name",
    "home_team.home_team_name",
    "away_team.away_team_name",
    "match_date"
).show(10, truncate=False)

+--------+----------------+-----------+----------------------+--------------+----------+
|match_id|competition_name|season_name|home_team_name        |away_team_name|match_date|
+--------+----------------+-----------+----------------------+--------------+----------+
|3825848 |La Liga         |2015/2016  |Levante UD            |Eibar         |2015-09-23|
|3825895 |La Liga         |2015/2016  |Las Palmas            |Sevilla       |2015-09-23|
|3825894 |La Liga         |2015/2016  |RC Deportivo La Coruña|Getafe        |2016-05-01|
|3825855 |La Liga         |2015/2016  |Málaga                |Levante UD    |2016-05-02|
|3825908 |La Liga         |2015/2016  |Espanyol              |Eibar         |2016-05-15|
|3825883 |La Liga         |2015/2016  |Málaga                |Las Palmas    |2016-05-15|
|3825900 |La Liga         |2015/2016  |Sporting Gijón        |Villarreal    |2016-05-15|
|3825902 |La Liga         |2015/2016  |Rayo Vallecano        |Levante UD    |2016-05-15|
|3825876 |La Liga    

In [14]:
from pyspark.sql.functions import input_file_name

events = spark.read.option(
    "multiline", "true"
).json("statsbomb-open-data/data/events/*.json") \
.withColumn("file", input_file_name())

26/07/16 15:47:23 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: statsbomb-open-data/data/events/*.json.
java.io.FileNotFoundException: File statsbomb-open-data/data/events/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apac

In [15]:
##vertex ve edge sayısı
vertices.count(), edges.count()

(13256, 208518)

In [16]:
## en çok pas alan ve atan oyuncular
from pyspark.sql.functions import sum

edges.groupBy("passer") \
     .agg(sum("pass_count").alias("total_passes")) \
     .orderBy(col("total_passes").desc()) \
     .show(20, truncate=False)

edges.groupBy("receiver") \
     .agg(sum("pass_count").alias("received_passes")) \
     .orderBy(col("received_passes").desc()) \
     .show(20, truncate=False)

+-------------------------------+------------+
|passer                         |total_passes|
+-------------------------------+------------+
|Lionel Andrés Messi Cuccittini |31881       |
|Sergio Busquets i Burgos       |28070       |
|Xavier Hernández Creus         |22618       |
|Gerard Piqué Bernabéu          |20969       |
|Andrés Iniesta Luján           |19989       |
|Jordi Alba Ramos               |19387       |
|Daniel Alves da Silva          |17604       |
|Javier Alejandro Mascherano    |11962       |
|Ivan Rakitić                   |11906       |
|Sergi Roberto Carnicer         |9455        |
|Carles Puyol i Saforcada       |8850        |
|Neymar da Silva Santos Junior  |8021        |
|Francesc Fàbregas i Soler      |7989        |
|Samuel Yves Umtiti             |7231        |
|Granit Xhaka                   |6735        |
|Eric-Sylvain Bilal Abidal      |6605        |
|Pedro Eliezer Rodríguez Ledesma|6525        |
|Keira Walsh                    |6218        |
|Víctor Valdé

+-------------------------------+---------------+
|receiver                       |received_passes|
+-------------------------------+---------------+
|Lionel Andrés Messi Cuccittini |42144          |
|Sergio Busquets i Burgos       |24817          |
|Xavier Hernández Creus         |22205          |
|Andrés Iniesta Luján           |21536          |
|Gerard Piqué Bernabéu          |17647          |
|Jordi Alba Ramos               |16831          |
|Daniel Alves da Silva          |15718          |
|Ivan Rakitić                   |11512          |
|Neymar da Silva Santos Junior  |11084          |
|Javier Alejandro Mascherano    |10075          |
|Luis Alberto Suárez Díaz       |9328           |
|Sergi Roberto Carnicer         |8705           |
|Pedro Eliezer Rodríguez Ledesma|8654           |
|Francesc Fàbregas i Soler      |8383           |
|Carles Puyol i Saforcada       |7164           |
|Antoine Griezmann              |6331           |
|Granit Xhaka                   |6172           |


In [17]:
## en fazla pas yapan ikililer
edges.orderBy(col("pass_count").desc()).show(20, truncate=False)

+---------+------------------------------+-----------+------------------------------+-------+---------+----------+
|passer_id|passer                        |receiver_id|receiver                      |team_id|team     |pass_count|
+---------+------------------------------+-----------+------------------------------+-------+---------+----------+
|5203     |Sergio Busquets i Burgos      |5503       |Lionel Andrés Messi Cuccittini|217    |Barcelona|4155      |
|4324     |Daniel Alves da Silva         |5503       |Lionel Andrés Messi Cuccittini|217    |Barcelona|3967      |
|20131    |Xavier Hernández Creus        |5503       |Lionel Andrés Messi Cuccittini|217    |Barcelona|3314      |
|5503     |Lionel Andrés Messi Cuccittini|20131      |Xavier Hernández Creus        |217    |Barcelona|2634      |
|20131    |Xavier Hernández Creus        |4324       |Daniel Alves da Silva         |217    |Barcelona|2615      |
|4324     |Daniel Alves da Silva         |20131      |Xavier Hernández Creus    

In [18]:
events.inputFiles()[:5]

['file:///Users/onurercen/Desktop/401proje/statsbomb-open-data/data/events/3825839.json',
 'file:///Users/onurercen/Desktop/401proje/statsbomb-open-data/data/events/3901257.json',
 'file:///Users/onurercen/Desktop/401proje/statsbomb-open-data/data/events/3825690.json',
 'file:///Users/onurercen/Desktop/401proje/statsbomb-open-data/data/events/69267.json',
 'file:///Users/onurercen/Desktop/401proje/statsbomb-open-data/data/events/3795220.json']

## Events match_id ile yeniden oku


In [19]:
from pyspark.sql.functions import input_file_name, regexp_extract

events = (
    spark.read
    .option("multiline", True)
    .json("statsbomb-open-data/data/events/*.json")
    .withColumn("file_path", input_file_name())
    .withColumn(
        "match_id",
        regexp_extract("file_path", r"(\d+)\.json$", 1).cast("int")
    )
)

26/07/16 15:49:00 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: statsbomb-open-data/data/events/*.json.
java.io.FileNotFoundException: File statsbomb-open-data/data/events/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apac

In [20]:
events.select("match_id").show(10)

+--------+
|match_id|
+--------+
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
+--------+
only showing top 10 rows


In [21]:
events.select("match_id").distinct().count()

4235

In [22]:
matches.select("match_id").distinct().count()

4235

numbers match

In [23]:
## Reconstruct passes dataframe with match_id

from pyspark.sql.functions import col

passes = (
    events
    .filter(col("type.name") == "Pass")
    .select(
        "match_id",
        col("player.id").alias("passer_id"),
        col("player.name").alias("passer"),
        col("pass.recipient.id").alias("receiver_id"),
        col("pass.recipient.name").alias("receiver"),
        col("team.id").alias("team_id"),
        col("team.name").alias("team"),
        "minute",
        "second"
    )
    .filter(col("receiver").isNotNull())
    .cache()
)

passes.count()   # cache'i doldurur

3847184

In [24]:
passes.printSchema()

root
 |-- match_id: integer (nullable = true)
 |-- passer_id: long (nullable = true)
 |-- passer: string (nullable = true)
 |-- receiver_id: long (nullable = true)
 |-- receiver: string (nullable = true)
 |-- team_id: long (nullable = true)
 |-- team: string (nullable = true)
 |-- minute: long (nullable = true)
 |-- second: long (nullable = true)



In [25]:
## Reconstruct edges dataframe with match_id
edges = (
    passes.groupBy(
        "match_id",
        "passer_id",
        "passer",
        "receiver_id",
        "receiver",
        "team_id",
        "team"
    )
    .count()
    .withColumnRenamed("count", "pass_count")
)

In [26]:
## Graph Validation

# Kaç oyuncu?
vertices.count()

# Kaç edge?
edges.count()

# Kaç maç?
edges.select("match_id").distinct().count()

4235

In [27]:
## En fazla pas yapan ikililer
edges.orderBy(
    col("pass_count").desc()
).show(20, truncate=False)

+--------+---------+--------------------------+-----------+------------------------------+-------+-------------------+----------+
|match_id|passer_id|passer                    |receiver_id|receiver                      |team_id|team               |pass_count|
+--------+---------+--------------------------+-----------+------------------------------+-------+-------------------+----------+
|3857255 |6765     |Rodrigo Hernández Cascante|6892       |Pau Francisco Torres          |772    |Spain              |69        |
|3869220 |6765     |Rodrigo Hernández Cascante|4353       |Aymeric Laporte               |772    |Spain              |68        |
|3857255 |6892     |Pau Francisco Torres      |6765       |Rodrigo Hernández Cascante    |772    |Spain              |67        |
|3775580 |4633     |Magdalena Lilly Eriksson  |4642       |Millie Bright                 |971    |Chelsea FCW        |63        |
|3893828 |4642     |Millie Bright             |10252      |Alex Greenwood                |

## Graph Analytics

In [38]:
from pyspark.sql.functions.builtin import countDistinct

out_degree = (
    edges
    .groupBy("passer_id", "passer")
    .agg(
        countDistinct("receiver_id").alias("out_degree")
    )
)

out_degree.orderBy("out_degree", ascending=False).show(20, truncate=False)

+---------+-------------------------------+----------+
|passer_id|passer                         |out_degree|
+---------+-------------------------------+----------+
|5503     |Lionel Andrés Messi Cuccittini |190       |
|5211     |Jordi Alba Ramos               |147       |
|5203     |Sergio Busquets i Burgos       |143       |
|6821     |Jesús Navas González           |126       |
|5470     |Ivan Rakitić                   |122       |
|5487     |Antoine Griezmann              |119       |
|5504     |Éver Maximiliano David Banega  |119       |
|5213     |Gerard Piqué Bernabéu          |115       |
|5201     |Sergio Ramos García            |114       |
|6867     |Papa Kouly Diop                |110       |
|6599     |Rubén Salvador Pérez Del Mármol|107       |
|6758     |Víctor Sánchez Mata            |103       |
|6720     |Pablo Sarabia García           |100       |
|6614     |Alexis Ruano Delgado           |100       |
|6913     |Fernando Navarro i Corbacho    |99        |
|26211    

In [39]:
##In-Degree Analysis
in_degree = (
    edges
    .groupBy("receiver_id", "receiver")
    .agg(
        countDistinct("passer_id").alias("in_degree")
    )
)

in_degree.orderBy("in_degree", ascending=False).show(20, truncate=False)

+-----------+-----------------------------------+---------+
|receiver_id|receiver                           |in_degree|
+-----------+-----------------------------------+---------+
|5503       |Lionel Andrés Messi Cuccittini     |204      |
|5203       |Sergio Busquets i Burgos           |141      |
|5211       |Jordi Alba Ramos                   |140      |
|6821       |Jesús Navas González               |135      |
|5487       |Antoine Griezmann                  |126      |
|5504       |Éver Maximiliano David Banega      |124      |
|5470       |Ivan Rakitić                       |118      |
|6720       |Pablo Sarabia García               |114      |
|6867       |Papa Kouly Diop                    |113      |
|6391       |Raúl García Escudero               |112      |
|5213       |Gerard Piqué Bernabéu              |110      |
|26211      |Joan Verdú Fernández               |109      |
|5201       |Sergio Ramos García                |105      |
|6651       |Joaquín Sánchez Rodríguez  

In [41]:
## TOTAL DEGREE
from pyspark.sql.functions import coalesce, col

degree = (
    out_degree.alias("o")
    .join(
        in_degree.alias("i"),
        col("o.passer_id") == col("i.receiver_id"),
        "full"
    )
    .select(
        coalesce(col("o.passer_id"), col("i.receiver_id")).alias("player_id"),
        coalesce(col("o.passer"), col("i.receiver")).alias("player"),
        coalesce(col("out_degree"), col("in_degree") * 0).alias("out_degree"),
        coalesce(col("in_degree"), col("out_degree") * 0).alias("in_degree")
    )
    .withColumn(
        "degree",
        col("out_degree") + col("in_degree")
    )
)

degree.orderBy(
    col("degree").desc()
).show(20, truncate=False)

+---------+-----------------------------------+----------+---------+------+
|player_id|player                             |out_degree|in_degree|degree|
+---------+-----------------------------------+----------+---------+------+
|5503     |Lionel Andrés Messi Cuccittini     |190       |204      |394   |
|5211     |Jordi Alba Ramos                   |147       |140      |287   |
|5203     |Sergio Busquets i Burgos           |143       |141      |284   |
|6821     |Jesús Navas González               |126       |135      |261   |
|5487     |Antoine Griezmann                  |119       |126      |245   |
|5504     |Éver Maximiliano David Banega      |119       |124      |243   |
|5470     |Ivan Rakitić                       |122       |118      |240   |
|5213     |Gerard Piqué Bernabéu              |115       |110      |225   |
|6867     |Papa Kouly Diop                    |110       |113      |223   |
|5201     |Sergio Ramos García                |114       |105      |219   |
|6720     |P